# 03 — Extract Raster Features to 1 km Grid (Revised)

Revised Google Colab notebook for raster-to-grid feature extraction.

Main revision:
- `exactextract` returns **GeoJSON Feature** objects by default, so extracted statistics are stored under the `properties` field.
- Notebook ini menambahkan fungsi `flatten_exactextract_result()` so `exactextract` outputs are parsed correctly.
- The main output is saved as a **GeoPackage (.gpkg)** for more efficient and stable spatial storage than GeoJSON.
- An Excel file is also produced for tabular auditing.


In [ ]:
# ============================================================
# 1. Install required libraries
# ============================================================

!pip install geopandas rasterio exactextract openpyxl pyogrio -q

import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import exactextract
from google.colab import drive, files


In [ ]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

drive.mount("/content/drive")


In [ ]:
# ============================================================
# 3. Set working directory and input paths
# ============================================================

BASE_DIR = "/content/drive/MyDrive/geospatial_biomass_carbon"

GRID_FILE = os.path.join(BASE_DIR, "grid_1km_jawa_barat.geojson")

SENTINEL_FILE = os.path.join(BASE_DIR, "sentinel2_indices_jawa_barat_2024_scale50.tif")
VIIRS_FILE = os.path.join(BASE_DIR, "viirs_nighttime_lights_jawa_barat_2024.tif")
LANDCOVER_FILE = os.path.join(BASE_DIR, "esa_worldcover_jawa_barat.tif")
CROPLAND_FILE = os.path.join(BASE_DIR, "cropland_mask_jawa_barat.tif")

OUTPUT_GPKG = os.path.join(BASE_DIR, "grid_1km_jawa_barat_with_raster_features.gpkg")
OUTPUT_EXCEL = os.path.join(BASE_DIR, "grid_1km_jawa_barat_with_raster_features.xlsx")

# Optional GeoJSON output. It is disabled by default because the file can be very large.
MAKE_GEOJSON = False
OUTPUT_GEOJSON = os.path.join(BASE_DIR, "grid_1km_jawa_barat_with_raster_features.geojson")

required_files = [
    GRID_FILE,
    SENTINEL_FILE,
    VIIRS_FILE,
    LANDCOVER_FILE,
    CROPLAND_FILE
]

for f in required_files:
    if not os.path.exists(f):
        raise FileNotFoundError(f"File not found: {f}")

print("All input files were found.")


In [ ]:
# ============================================================
# 4. Load grid layer
# ============================================================

grid = gpd.read_file(GRID_FILE).to_crs("EPSG:4326")

print("Grid loaded:", grid.shape)
print("Grid CRS:", grid.crs)
display(grid.head())


In [ ]:
# ============================================================
# 5. Helper functions for exactextract outputs
# ============================================================

grid_crs_cache = {}

def get_projected_grid(grid_gdf, raster_path):
    """
    Reproject the grid to the raster CRS.
    The result is cached to avoid repeated reprojection.
    """
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs

    if raster_crs is None:
        raise ValueError(f"Raster has no CRS: {raster_path}")

    crs_string = raster_crs.to_string()

    if crs_string not in grid_crs_cache:
        print(f"  -> Reprojecting grid to {crs_string}")
        grid_crs_cache[crs_string] = grid_gdf.to_crs(raster_crs)

    return grid_crs_cache[crs_string]


def flatten_exactextract_result(stats):
    """
    exactextract sering mengembalikan output default dalam bentuk GeoJSON Feature:
    [
      {"type": "Feature", "properties": {...}},
      ...
    ]

    If langsung pd.DataFrame(stats), columnnya menjadi ['type', 'properties'].
    Fungsi ini mengambil isi 'properties' dan mengubahnya menjadi DataFrame datar.
    """
    rows = []

    for item in stats:
        if isinstance(item, dict) and "properties" in item:
            props = item.get("properties", {})
        elif isinstance(item, dict):
            props = item
        else:
            props = {}

        flat = {}

        for key, value in props.items():
            # If nilai berupa list/array, pecah menjadi beberapa column
            if isinstance(value, (list, tuple, np.ndarray)):
                for i, v in enumerate(value, start=1):
                    flat[f"{key}_{i}"] = v
            else:
                flat[key] = value

        rows.append(flat)

    return pd.DataFrame(rows)


def standardize_multiband_columns(raw_df, band_names, stat_name="mean"):
    """
    Menyamakan nama column exactextract multiband menjadi band_names.
    Mendukung beberapa kemungkinan format output:
    - mean_1, mean_2, ...
    - mean.band_1, mean.band_2, ...
    - band_1_mean, band_2_mean, ...
    - atau column numerik several bands.
    """
    df = raw_df.copy()

    # Drop non-statistical columns when present
    drop_candidates = ["id", "fid", "index", "index_right"]
    df = df.drop(columns=[c for c in drop_candidates if c in df.columns], errors="ignore")

    # Case 1: jumlah column sudah sama dengan jumlah band
    if len(df.columns) == len(band_names):
        df.columns = band_names
        return df

    # Kasus 2: mean_1, mean_2, dst.
    cols_mean_i = [f"{stat_name}_{i}" for i in range(1, len(band_names) + 1)]
    if all(c in df.columns for c in cols_mean_i):
        out = df[cols_mean_i].copy()
        out.columns = band_names
        return out

    # Kasus 3: mean.1, mean.2, dst.
    cols_mean_dot_i = [f"{stat_name}.{i}" for i in range(1, len(band_names) + 1)]
    if all(c in df.columns for c in cols_mean_dot_i):
        out = df[cols_mean_dot_i].copy()
        out.columns = band_names
        return out

    # Kasus 4: column mengandung kata mean dan jumlahnya sama dengan band
    stat_cols = [c for c in df.columns if stat_name.lower() in str(c).lower()]
    if len(stat_cols) == len(band_names):
        out = df[stat_cols].copy()
        out.columns = band_names
        return out

    # Kasus 5: column numerik saja, jumlah sama dengan band
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == len(band_names):
        out = df[numeric_cols].copy()
        out.columns = band_names
        return out

    print("Detected exactextract columns:")
    print(df.columns.tolist())
    print("Preview raw_df:")
    display(df.head())

    raise ValueError(
        "The multiband exactextract output columns were not recognized. "
        "Inspect the printed columns above if the error persists."
    )


def extract_multiband_exact(grid_gdf, raster_path, band_names, stat_name="mean"):
    """
    Extract mean values from a multiband raster using exactextract.
    """
    print(f"\\nProcessing multiband raster: {os.path.basename(raster_path)}")

    with rasterio.open(raster_path) as src:
        print("Raster CRS:", src.crs)
        print("Band count:", src.count)
        print("Raster nodata:", src.nodata)

        if src.count != len(band_names):
            raise ValueError(
                f"The number of raster bands ({src.count}) does not match the number of band names ({len(band_names)})."
            )

    grid_proj = get_projected_grid(grid_gdf, raster_path)

    stats = exactextract.exact_extract(
        raster_path,
        grid_proj,
        [stat_name]
    )

    raw_df = flatten_exactextract_result(stats)
    result_df = standardize_multiband_columns(raw_df, band_names, stat_name=stat_name)
    result_df.index = grid_gdf.index

    return result_df


def extract_singleband_exact(grid_gdf, raster_path, output_col, stat_type="mean"):
    """
    Extract statistics from a single-band raster.
    For cropland: mean
    For VIIRS: mean
    For land cover: majority
    """
    print(f"\\nProcessing singleband raster: {os.path.basename(raster_path)} | Stat: {stat_type}")

    with rasterio.open(raster_path) as src:
        print("Raster CRS:", src.crs)
        print("Band count:", src.count)
        print("Raster nodata:", src.nodata)

    grid_proj = get_projected_grid(grid_gdf, raster_path)

    stats = exactextract.exact_extract(
        raster_path,
        grid_proj,
        [stat_type]
    )

    raw_df = flatten_exactextract_result(stats)
    raw_df.index = grid_gdf.index

    if stat_type in raw_df.columns:
        return raw_df[stat_type].rename(output_col)

    # Fallback for other column names containing `stat_type`
    stat_cols = [c for c in raw_df.columns if stat_type.lower() in str(c).lower()]
    if len(stat_cols) == 1:
        return raw_df[stat_cols[0]].rename(output_col)

    # Fallback when only one numeric column is available
    numeric_cols = raw_df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 1:
        return raw_df[numeric_cols[0]].rename(output_col)

    print("Detected singleband exactextract columns:")
    print(raw_df.columns.tolist())
    display(raw_df.head())

    raise ValueError(
        f"The exactextract output column for {output_col} was not recognized."
    )


def minmax(series):
    """
    Normalisasi min-max 0–1.
    """
    s = pd.to_numeric(series, errors="coerce")

    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=series.index)

    if s.max() == s.min():
        return pd.Series(0, index=series.index)

    return (s - s.min()) / (s.max() - s.min())


In [ ]:
# ============================================================
# 6. Extract Sentinel-2 multiband features
# ============================================================

sentinel_band_names = [
    "mean_B2",
    "mean_B3",
    "mean_B4",
    "mean_B8",
    "mean_B11",
    "mean_B12",
    "mean_NDVI",
    "mean_EVI",
    "mean_NDWI",
    "mean_NBR",
    "mean_SAVI"
]

sentinel_features = extract_multiband_exact(
    grid_gdf=grid,
    raster_path=SENTINEL_FILE,
    band_names=sentinel_band_names,
    stat_name="mean"
)

print("Sentinel features extracted:", sentinel_features.shape)
display(sentinel_features.head())


In [ ]:
# ============================================================
# 7. Extract raster singleband features
# ============================================================

viirs_feature = extract_singleband_exact(
    grid_gdf=grid,
    raster_path=VIIRS_FILE,
    output_col="mean_VIIRS_NTL",
    stat_type="mean"
)

cropland_feature = extract_singleband_exact(
    grid_gdf=grid,
    raster_path=CROPLAND_FILE,
    output_col="cropland_ratio",
    stat_type="mean"
)

cropland_feature = cropland_feature.clip(lower=0, upper=1)

landcover_feature = extract_singleband_exact(
    grid_gdf=grid,
    raster_path=LANDCOVER_FILE,
    output_col="landcover_majority",
    stat_type="majority"
)

print("Singleband features extracted.")


In [ ]:
# ============================================================
# 8. Merge extracted raster features
# ============================================================

grid_features = grid.copy()

grid_features = pd.concat(
    [
        grid_features,
        sentinel_features
    ],
    axis=1
)

grid_features["mean_VIIRS_NTL"] = viirs_feature
grid_features["cropland_ratio"] = cropland_feature
grid_features["landcover_majority"] = landcover_feature

print("All raster features were successfully merged.")
display(grid_features.head())


In [ ]:
# ============================================================
# 9. Interpret ESA WorldCover classes and preliminary scores
# ============================================================

esa_worldcover_classes = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare or sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss and lichen"
}

grid_features["landcover_majority"] = pd.to_numeric(
    grid_features["landcover_majority"],
    errors="coerce"
)

grid_features["landcover_name"] = grid_features["landcover_majority"].map(
    esa_worldcover_classes
)

grid_features["ndvi_score"] = minmax(grid_features["mean_NDVI"])
grid_features["ntl_score"] = minmax(grid_features["mean_VIIRS_NTL"])
grid_features["cropland_score"] = grid_features["cropland_ratio"].fillna(0)

# Built-up and water classes are less suitable as feedstock areas.
grid_features["environmental_score"] = np.where(
    grid_features["landcover_majority"].isin([50, 80]),
    0,
    1
)

display(grid_features[[
    "grid_id",
    "district",
    "mean_NDVI",
    "mean_VIIRS_NTL",
    "cropland_ratio",
    "landcover_majority",
    "landcover_name",
    "ndvi_score",
    "ntl_score",
    "cropland_score",
    "environmental_score"
]].head())


In [ ]:
# ============================================================
# 10. Missing value check
# ============================================================

feature_cols = [
    "mean_B2",
    "mean_B3",
    "mean_B4",
    "mean_B8",
    "mean_B11",
    "mean_B12",
    "mean_NDVI",
    "mean_EVI",
    "mean_NDWI",
    "mean_NBR",
    "mean_SAVI",
    "mean_VIIRS_NTL",
    "cropland_ratio",
    "landcover_majority",
    "ndvi_score",
    "ntl_score",
    "cropland_score",
    "environmental_score"
]

missing_summary = (
    grid_features[feature_cols]
    .isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["feature", "missing_count"]

print("Missing summary:")
display(missing_summary)

summary_district = (
    grid_features
    .groupby("district")
    .agg(
        grid_count=("grid_id", "count"),
        mean_NDVI=("mean_NDVI", "mean"),
        mean_EVI=("mean_EVI", "mean"),
        mean_NDWI=("mean_NDWI", "mean"),
        mean_NBR=("mean_NBR", "mean"),
        mean_VIIRS_NTL=("mean_VIIRS_NTL", "mean"),
        mean_cropland_ratio=("cropland_ratio", "mean"),
        mean_environmental_score=("environmental_score", "mean")
    )
    .reset_index()
    .sort_values("district")
)

display(summary_district.head())


In [ ]:
# ============================================================
# 11. Save outputs
# ============================================================

# Main output for QGIS and downstream spatial processing
grid_features.to_file(
    OUTPUT_GPKG,
    layer="grid_raster_features",
    driver="GPKG"
)

print("GeoPackage saved:")
print(OUTPUT_GPKG)

# Excel output for tabular auditing
excel_df = grid_features.copy()
excel_df["geometry_wkt"] = excel_df.geometry.astype(str)
excel_df = excel_df.drop(columns="geometry")

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    excel_df.to_excel(writer, sheet_name="Grid_Raster_Features", index=False)
    summary_district.to_excel(writer, sheet_name="Summary_District", index=False)
    missing_summary.to_excel(writer, sheet_name="Missing_Check", index=False)

print("Excel file saved:")
print(OUTPUT_EXCEL)

# Optional GeoJSON output, only when required.
if MAKE_GEOJSON:
    grid_features.to_file(OUTPUT_GEOJSON, driver="GeoJSON")
    print("GeoJSON saved:")
    print(OUTPUT_GEOJSON)

files.download(OUTPUT_EXCEL)

print("Completed. GeoPackage and Excel files were saved to Google Drive.")
